# Phase 1 · Audit of the v1 results (R1 revision)

This notebook analyses the **existing** v1 results (ART-001, produced by Model A = ART-008) to answer the reviewer requests that do not need new experiments, and to document the problems that the v2 run must fix.

| Section | Reviewer request / audit item | Table(s) | Figure |
|---|---|---|---|
| 1 | R1-a, R1-b, R2-M1: classifier performance, threshold, calibration | T1, T1b | F1 |
| 2 | R1-c: stratification by prediction correctness | T2, T3 | F2 |
| 3 | A-03: patient-level Grad-CAM vs Eigen-CAM comparison | T4 | F3 |
| 4 | A-13: degradation kinetics and slopes with CIs | T5, T6 | F4 |
| 5 | localization under perturbation (TP primary) | T7 | F5 |
| 6 | R2-M2: percentile FAR is definitional | T8 | F6 |
| 7 | A-04, A-05: component redundancy and validity against localization change | T9, T9b | F7 |
| 8 | A-06: baselines | T10 | |
| 9 | A-01: manuscript vs v1 data | T11 | |

**Important.** These numbers describe the v1 data. They go into the response letter and the audit record. The manuscript figures are regenerated in Phase 5 from the v2 run with the same plotting and statistics code (`src/r1lib`).

Outputs: tables in `output/phase1/`, figures in `fig/phase1_v1/` (PNG + PDF at 300 dpi, plus the CSV of the plotted data).

## Configuration cell

All paths and constants in one place. The input is ART-001 (v1 CSV with the `xfi` column); its hash is checked before anything is computed.
Bootstrap: B = 2000 patient resamples, percentile 95% CIs, fixed seed.

In [ ]:
import sys, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from r1lib import style as S, stats as ST
S.apply_style()

CFG = yaml.safe_load(open(ROOT / "configs" / "config.yaml"))
SEED = CFG["project"]["seed"]
B = CFG["metrics"]["bootstrap_B"]
ALPHA = CFG["metrics"]["alpha"]

IN_CSV = ROOT / CFG["paths"]["v1_legacy_csv"]
IN_SHA = "ef090d4706a40ef2bd4821f51028921484d84811ab67dd853ff4c477f1ed4b23"   # ART-001
OUT = ROOT / "output" / "phase1"; OUT.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "fig" / "phase1_v1"; FIG.mkdir(parents=True, exist_ok=True)
PRODUCED = []          # every file written by this notebook (for ARTIFACTS)

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def save_table(df, name):
    p = OUT / f"{name}.csv"; df.to_csv(p, index=False); PRODUCED.append(p); return p

def save_fig(fig, stem, data):
    PRODUCED.extend(S.save_figure(fig, stem, data, FIG))

print("input:", IN_CSV)
print("B =", B, "| seed =", SEED, "| out:", OUT, "| fig:", FIG)

## Load and prepare

Loads ART-001, verifies its hash, and adds two patient-level columns:
- `pred`: Model A prediction at the v1 operating threshold (prob > 0.5), the same rule used to choose the CAM target class in v1;
- `group`: TP, FN (positives) or TN, FP (negatives).

`xai` keeps only Grad-CAM and Eigen-CAM; `pos` keeps the positive patients of `xai` (IoU is defined only for them).

In [ ]:
assert sha256(IN_CSV) == IN_SHA, "input CSV is not ART-001"
df = pd.read_csv(IN_CSV)
df["pred"] = (df["Model_Prob"] > 0.5).astype(int)
df["group"] = np.select([(df.Label == 1) & (df.pred == 1), (df.Label == 1) & (df.pred == 0),
                         (df.Label == 0) & (df.pred == 0)], ["TP", "FN", "TN"], "FP")
xai = df[df.Method.isin(S.METHODS)].copy()
pos = xai[xai.Label == 1].copy()
pt = df.drop_duplicates("Patient")[["Patient", "Label", "Model_Prob", "pred", "group"]].reset_index(drop=True)
print(f"rows={len(df)}  patients={len(pt)}  positives={int(pt.Label.sum())}")
print(pt.group.value_counts().reindex(["TP", "FN", "TN", "FP"]).to_string())

## 1 · Classifier performance of Model A (R1-a, R1-b, R2-M1)

**T1** reports, on the 5,337 test patients, AUC, average precision, expected calibration error (10 equal-width bins) and Brier score, plus accuracy, precision, recall, specificity, F1 and balanced accuracy at two thresholds:
- `0.5`: the operating point used in v1 (and for the CAM target class);
- `youden_posthoc`: the Youden-optimal threshold **chosen on the test set**. It is reported only to show how much the 0.5 threshold costs; it is optimistic and is not used for any XAI result. In v2, Model B chooses its threshold on a validation split (D-07).

All CIs are patient bootstrap percentile intervals. **T1b** gives the confusion counts at both thresholds.
This cell takes about one minute (bootstrap of 16 metrics).

In [ ]:
y, p = pt.Label.values, pt.Model_Prob.values
thr_y = ST.youden_threshold(y, p)
T1 = ST.classifier_table(y, p, {"0.5": 0.5, "youden_posthoc": thr_y}, B=B, seed=SEED, alpha=ALPHA)
T1.insert(0, "model", "A (ART-008)")
save_table(T1, "T1_classifier_metrics")

T1b = pd.DataFrame([{"threshold": k, "value": v, **{c: ST.threshold_metrics(y, p, v)[c] for c in ["TP", "FN", "TN", "FP"]}}
                    for k, v in {"0.5": 0.5, "youden_posthoc": thr_y}.items()])
save_table(T1b, "T1b_confusion_counts")

with pd.option_context("display.float_format", "{:.4f}".format):
    print(T1.to_string(index=False)); print(); print(T1b.to_string(index=False))
print(f"\nYouden threshold (post hoc, test set): {thr_y:.4f}   prevalence: {y.mean():.4f}")

### Figure F1 · Classifier diagnostics

(a) ROC curve with the two operating points; (b) precision-recall curve with the prevalence line; (c) reliability diagram (10 bins; marker area proportional to bin size); (d) distribution of predicted probabilities by true class with the two thresholds.
The plotted data are saved as `F1_classifier__roc.csv`, `__pr.csv`, `__reliability.csv` and `__hist.csv`.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
fpr, tpr, _ = roc_curve(y, p); prc, rcl, _ = precision_recall_curve(y, p)
rel = ST.reliability_curve(y, p, 10)
bins = np.linspace(0, 1, 41)
h_pos, _ = np.histogram(p[y == 1], bins=bins, density=True); h_neg, _ = np.histogram(p[y == 0], bins=bins, density=True)
ops = {k: ST.threshold_metrics(y, p, v) for k, v in {"0.5": 0.5, "Youden": thr_y}.items()}

fig, ax = plt.subplots(2, 2, figsize=(7.2, 6.6))
a = ax[0, 0]
a.plot(fpr, tpr, color=S.INK, lw=1.8); a.plot([0, 1], [0, 1], ls="--", color=S.NEUTRAL, lw=1)
for (k, m), mk in zip(ops.items(), ["o", "D"]):
    a.plot(1 - m["specificity"], m["recall"], mk, ms=7, mfc="white", mec=S.INK, mew=1.5, label=f"threshold {k}")
a.text(0.97, 0.05, f"AUC = {roc_auc_score(y, p):.3f}", ha="right", transform=a.transAxes, color=S.INK)
a.set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.01)); a.legend(loc="center right")
S.panel_label(a, "a")

a = ax[0, 1]
a.plot(rcl, prc, color=S.INK, lw=1.8); a.axhline(y.mean(), ls="--", color=S.NEUTRAL, lw=1)
for (k, m), mk in zip(ops.items(), ["o", "D"]):
    a.plot(m["recall"], m["precision"], mk, ms=7, mfc="white", mec=S.INK, mew=1.5)
a.text(0.97, 0.92, f"AP = {average_precision_score(y, p):.3f}", ha="right", transform=a.transAxes, color=S.INK)
a.text(0.02, y.mean() + 0.02, f"prevalence {y.mean():.3f}", color=S.INK2, fontsize=7)
a.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.01)); S.panel_label(a, "b")

a = ax[1, 0]
a.plot([0, 1], [0, 1], ls="--", color=S.NEUTRAL, lw=1)
a.plot(rel.mean_pred, rel.frac_pos, color=S.INK, lw=1.5)
a.scatter(rel.mean_pred, rel.frac_pos, s=20 + 400 * rel.n / rel.n.max(), color="white", edgecolor=S.INK, lw=1.5, zorder=3)
a.text(0.03, 0.92, f"ECE = {ST.ece_score(y, p):.3f}\nBrier = {ST.brier(y, p):.3f}", transform=a.transAxes, va="top", color=S.INK)
a.set(xlabel="Mean predicted probability", ylabel="Observed fraction positive", xlim=(0, 1), ylim=(0, 1)); S.panel_label(a, "c")

a = ax[1, 1]
a.stairs(h_neg, bins, color=S.NEUTRAL, lw=1.8, label=f"negative (n={int((y == 0).sum())})")
a.stairs(h_pos, bins, color=S.INK, lw=1.8, label=f"positive (n={int(y.sum())})")
a.axvline(0.5, color=S.INK2, lw=1, ls="-"); a.axvline(thr_y, color=S.INK2, lw=1, ls=":")
a.text(0.5, a.get_ylim()[1] * 0.97, " 0.5", color=S.INK2, fontsize=7, va="top")
a.text(thr_y, a.get_ylim()[1] * 0.88, " Youden", color=S.INK2, fontsize=7, va="top")
a.set(xlabel="Predicted probability of pneumonia", ylabel="Density", xlim=(0, 1)); a.legend(loc="center right")
S.panel_label(a, "d")
fig.subplots_adjust(hspace=0.55, wspace=0.3)

save_fig(fig, "F1_classifier", {
    "roc": pd.DataFrame({"fpr": fpr, "tpr": tpr}),
    "pr": pd.DataFrame({"recall": rcl, "precision": prc}),
    "reliability": rel,
    "hist": pd.DataFrame({"bin_low": bins[:-1], "bin_high": bins[1:], "density_negative": h_neg, "density_positive": h_pos})})
print("saved F1_classifier")

## 2 · Stratification by prediction correctness (R1-c)

The CAM target class in v1 is the model's predicted class. For a false-negative patient (positive, predicted negative) Grad-CAM explains the "no pneumonia" output, so its overlap with the lesion box is not expected to be high. Eigen-CAM ignores the target class in `grad-cam` 1.5.7, so it is affected only through the image itself.

- **T2**: clean-image IoU (relative 90/80, absolute 0.5/0.7) per method for all positives, TP and FN, one value per patient, bootstrap CI.
- **T3**: mean XFI per patient (over the 9 perturbation conditions) for TP, FN, TN, FP, bootstrap CI.

In [ ]:
iou_cols = {"iou_c_rel_90": "Rel-90", "iou_c_rel_80": "Rel-80", "iou_c_abs_5": "Abs-0.5", "iou_c_abs_7": "Abs-0.7"}
clean = pos.drop_duplicates(["Patient", "Method"])           # clean IoU is identical across conditions
rows = []
for m in S.METHODS:
    for g, sub in [("all positives", clean[clean.Method == m]), ("TP", clean[(clean.Method == m) & (clean.group == "TP")]),
                   ("FN", clean[(clean.Method == m) & (clean.group == "FN")])]:
        for c, lab in iou_cols.items():
            est, lo, hi = ST.boot_ci_stat(sub[c].values, B=B, seed=SEED)
            rows.append(dict(method=m, group=g, n=len(sub), iou=lab, mean=est, ci_low=lo, ci_high=hi))
T2 = pd.DataFrame(rows); save_table(T2, "T2_clean_iou_by_correctness")

pp_x = xai.groupby(["Patient", "Method", "group"], as_index=False)["xfi"].mean()
rows = []
for m in S.METHODS:
    for g in ["TP", "FN", "TN", "FP"]:
        v = pp_x[(pp_x.Method == m) & (pp_x.group == g)]["xfi"].values
        est, lo, hi = ST.boot_ci_stat(v, B=B, seed=SEED)
        med = float(np.median(v))
        rows.append(dict(method=m, group=g, n=len(v), mean_xfi=est, ci_low=lo, ci_high=hi, median_xfi=med))
T3 = pd.DataFrame(rows); save_table(T3, "T3_xfi_by_correctness")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(T2.pivot_table(index=["method", "group"], columns="iou", values="mean").to_string()); print(); print(T3.to_string(index=False))

### Figure F2 · Localization and fragility by prediction correctness

(a) Clean-image Rel-90 IoU for all positives, TP and FN; (b) patient-level mean XFI for TP, FN, TN and FP. Points are means, bars are 95% bootstrap CIs.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0))
dx = {"GradCAM": -0.12, "EigenCAM": 0.12}
a = ax[0]; groups = ["all positives", "TP", "FN"]
d2 = T2[T2.iou == "Rel-90"]
for m in S.METHODS:
    s = d2[d2.method == m].set_index("group").loc[groups]
    x = np.arange(len(groups)) + dx[m]
    a.errorbar(x, s["mean"], yerr=[s["mean"] - s.ci_low, s.ci_high - s["mean"]], fmt=S.METHOD_MARKERS[m], ms=6,
               color=S.METHOD_COLORS[m], capsize=3, lw=1.5, label=S.METHOD_LABELS[m])
a.set_xticks(range(len(groups)), [f"{g}\n(n={int(d2[d2.group == g].n.iloc[0])})" for g in groups])
a.set(ylabel="Clean-image IoU (Rel-90)", ylim=(0, None)); a.legend(loc="lower left"); a.grid(axis="x", visible=False)
S.panel_label(a, "a")
a = ax[1]; groups = ["TP", "FN", "TN", "FP"]
for m in S.METHODS:
    s = T3[T3.method == m].set_index("group").loc[groups]
    x = np.arange(len(groups)) + dx[m]
    a.errorbar(x, s.mean_xfi, yerr=[s.mean_xfi - s.ci_low, s.ci_high - s.mean_xfi], fmt=S.METHOD_MARKERS[m], ms=6,
               color=S.METHOD_COLORS[m], capsize=3, lw=1.5, label=S.METHOD_LABELS[m])
a.set_xticks(range(len(groups)), [f"{g}\n(n={int(T3[T3.group == g].n.iloc[0])})" for g in groups])
a.set(ylabel="Mean XFI per patient", ylim=(0, None)); a.grid(axis="x", visible=False)
S.panel_label(a, "b")
fig.subplots_adjust(wspace=0.32, bottom=0.2)
save_fig(fig, "F2_correctness", {"iou": d2, "xfi": T3})
print("saved F2_correctness")

## 3 · Grad-CAM vs Eigen-CAM with the patient as the unit (A-03)

v1 ran one Wilcoxon test on 48,033 rows (9 rows per patient), which treats repeated measurements of the same patient as independent.
Here:
- **overall**: one value per patient (mean XFI over the 9 conditions), paired Wilcoxon, n = 5,337;
- **per condition**: one paired test per perturbation type and level (9 tests), Holm-corrected;
- for every comparison: mean and median paired difference (Grad-CAM minus Eigen-CAM) with bootstrap CIs, share of patients where Grad-CAM is less fragile, and effect size r = |z| / sqrt(n);
- the v1 row-level test is repeated for reference (it is what the manuscript reported).

Negative difference = Grad-CAM less fragile.

In [ ]:
def compare(a, b, label, typ, lvl):
    d = a - b
    mean_e, mean_lo, mean_hi = ST.boot_ci_stat(d, np.mean, B=B, seed=SEED)
    med_e, med_lo, med_hi = ST.boot_ci_stat(d, np.median, B=B, seed=SEED)
    w = ST.wilcoxon_paired(a, b)
    return dict(comparison=label, type=typ, level=lvl, n=len(d), mean_diff=mean_e, mean_ci_low=mean_lo, mean_ci_high=mean_hi,
                median_diff=med_e, median_ci_low=med_lo, median_ci_high=med_hi, share_grad_lower=float((d < 0).mean()),
                W=w["W"], p=w["p"], z=w["z"], r=w["r"])

rows = []
pp = xai.groupby(["Patient", "Method"])["xfi"].mean().unstack()
rows.append(compare(pp.GradCAM.values, pp.EigenCAM.values, "patient mean over 9 conditions", "all", "all"))
for t in S.TYPES:
    for l in [1, 2, 3]:
        q = xai[(xai.Type == t) & (xai.Level == l)].pivot_table(index="Patient", columns="Method", values="xfi")
        rows.append(compare(q.GradCAM.values, q.EigenCAM.values, "per condition", t, l))
T4 = pd.DataFrame(rows)
cond = T4.comparison == "per condition"
T4["p_holm"] = np.nan; T4.loc[cond, "p_holm"] = ST.holm(T4.loc[cond, "p"].values)

rl = xai.pivot_table(index=["Patient", "Type", "Level"], columns="Method", values="xfi")
w_rl = ST.wilcoxon_paired(rl.GradCAM.values, rl.EigenCAM.values)
T4 = pd.concat([T4, pd.DataFrame([dict(comparison="v1 row-level (pseudo-replicated)", type="all", level="all", n=w_rl["n"],
                                       mean_diff=float((rl.GradCAM - rl.EigenCAM).mean()), W=w_rl["W"], p=w_rl["p"], z=w_rl["z"], r=w_rl["r"])])],
               ignore_index=True)
save_table(T4, "T4_grad_vs_eigen_patient_level")
with pd.option_context("display.float_format", "{:.4g}".format):
    print(T4[["comparison", "type", "level", "n", "mean_diff", "median_diff", "median_ci_low", "median_ci_high",
              "share_grad_lower", "p", "p_holm", "r"]].to_string(index=False))

### Figure F3 · Paired difference in XFI (Grad-CAM minus Eigen-CAM)

Median paired difference with 95% bootstrap CI, one row per condition plus the patient-level overall comparison. Values left of zero: Grad-CAM less fragile. The Holm-adjusted p-value is printed at the right of each row.

In [ ]:
f3 = T4[T4.comparison != "v1 row-level (pseudo-replicated)"].copy()
f3["row"] = np.where(f3.type == "all", "All conditions (patient mean)",
                     f3.type.map(S.TYPE_LABELS) + ", level " + f3.level.astype(str))
f3 = f3.iloc[::-1].reset_index(drop=True)
fig, a = plt.subplots(figsize=(7.2, 3.6))
yy = np.arange(len(f3))
for i, r in f3.iterrows():
    col = S.INK if r.type == "all" else S.INK2
    a.errorbar(r.median_diff, i, xerr=[[r.median_diff - r.median_ci_low], [r.median_ci_high - r.median_diff]],
               fmt="D" if r.type == "all" else "o", ms=6, color=col, mfc="white" if r.type != "all" else col, capsize=3, lw=1.5)
    fmt = lambda v: "< 1e-300" if v == 0 else f"= {v:.1e}"
    ptxt = f"p_Holm {fmt(r.p_holm)}" if r.type != "all" else f"p = {r.p:.3f}"
    a.text(1.01, i, ptxt, transform=a.get_yaxis_transform(), va="center", fontsize=7, color=S.INK2)
a.axvline(0, color=S.INK, lw=1)
a.set_yticks(yy, f3.row); a.set_xlabel("Median paired difference in XFI (Grad-CAM − Eigen-CAM)")
a.grid(axis="y", visible=False)
xl = max(abs(f3.median_ci_low).max(), abs(f3.median_ci_high).max()) * 1.15; a.set_xlim(-xl, xl)
a.text(0.02, 1.0, "← Grad-CAM less fragile", transform=a.transAxes, fontsize=7, color=S.INK2, va="bottom")
a.text(0.98, 1.0, "Eigen-CAM less fragile →", transform=a.transAxes, fontsize=7, color=S.INK2, va="bottom", ha="right")
save_fig(fig, "F3_paired_difference", f3.drop(columns=["row"]).assign(row_label=f3.row))
print("saved F3_paired_difference")

## 4 · Degradation kinetics (A-13)

- **T5**: mean XFI per method, perturbation type and level (and pooled over types), patient bootstrap CI.
- **T6**: degradation slope = OLS slope of XFI on level (1 to 3), computed **per patient**, then averaged; per type and overall (overall slope uses the per-patient mean over types at each level). The Grad-CAM vs Eigen-CAM difference in slopes is tested with paired Wilcoxon (Holm over the 4 tests).

v1 manuscript reported negative slopes; since XFI rises with severity, slopes are positive here.

In [ ]:
rows = []
for m in S.METHODS:
    for t in S.TYPES + ["pooled"]:
        for l in [1, 2, 3]:
            s = xai[(xai.Method == m) & (xai.Level == l)]
            v = (s[s.Type == t].xfi.values if t != "pooled" else s.groupby("Patient").xfi.mean().values)
            est, lo, hi = ST.boot_ci_stat(v, B=B, seed=SEED)
            rows.append(dict(method=m, type=t, level=l, n=len(v), mean_xfi=est, ci_low=lo, ci_high=hi, median_xfi=float(np.median(v))))
T5 = pd.DataFrame(rows); save_table(T5, "T5_xfi_by_type_level")

sl_t = ST.per_patient_slope(xai, by=("Patient", "Method", "Type"))
pooled = xai.groupby(["Patient", "Method", "Level"], as_index=False).xfi.mean()
sl_o = ST.per_patient_slope(pooled, by=("Patient", "Method")).assign(Type="pooled")
sl = pd.concat([sl_t, sl_o], ignore_index=True)
rows = []
for t in S.TYPES + ["pooled"]:
    q = sl[sl.Type == t].pivot_table(index="Patient", columns="Method", values="slope")
    rec = dict(type=t, n=len(q))
    for m in S.METHODS:
        e, lo, hi = ST.boot_ci_stat(q[m].values, B=B, seed=SEED); rec.update({f"{m}_slope": e, f"{m}_ci_low": lo, f"{m}_ci_high": hi})
    d = ST.boot_ci_stat((q.GradCAM - q.EigenCAM).values, B=B, seed=SEED)
    w = ST.wilcoxon_paired(q.GradCAM.values, q.EigenCAM.values)
    rec.update(diff=d[0], diff_ci_low=d[1], diff_ci_high=d[2], p=w["p"], r=w["r"]); rows.append(rec)
T6 = pd.DataFrame(rows); T6["p_holm"] = ST.holm(T6.p.values)
save_table(T6, "T6_degradation_slopes"); save_table(sl, "T6b_per_patient_slopes")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(T5[T5.type == "pooled"].to_string(index=False)); print(); print(T6.to_string(index=False))

### Figure F4 · XFI against perturbation severity

One panel per perturbation type: mean XFI per level with the 95% bootstrap CI as a band. The shared y-axis makes the types directly comparable.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.7), sharey=True)
for a, t, letter in zip(ax, S.TYPES, "abc"):
    for m in S.METHODS:
        s = T5[(T5.method == m) & (T5.type == t)]
        a.fill_between(s.level, s.ci_low, s.ci_high, color=S.METHOD_COLORS[m], alpha=0.18, lw=0)
        a.plot(s.level, s.mean_xfi, marker=S.METHOD_MARKERS[m], color=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m])
    a.set_xticks([1, 2, 3], ["L1", "L2", "L3"]); a.set_xlabel(S.TYPE_LABELS[t]); a.grid(axis="x", visible=False)
    S.panel_label(a, letter)
ax[0].set_ylabel("Mean XFI"); ax[0].set_ylim(0, None); ax[0].legend(loc="upper left")
fig.subplots_adjust(wspace=0.12, bottom=0.22)
save_fig(fig, "F4_xfi_kinetics", T5[T5.type != "pooled"])
print("saved F4_xfi_kinetics")

## 5 · Localization under perturbation

For positive patients, Rel-90 IoU against the ground-truth box on the clean image and on each perturbed image. **TP is the primary population** (D-12); all positives are kept in the table.
Relative drop = 1 - mean(noisy IoU) / mean(clean IoU), with a paired patient bootstrap CI (ratio of means, so patients with zero clean IoU do not produce infinite ratios; the v1 figure clipped per-sample ratios at zero, which biases the drop upwards).

In [ ]:
rows = []
for popname, popdf in [("TP", pos[pos.group == "TP"]), ("all positives", pos)]:
    for m in S.METHODS:
        for t in S.TYPES:
            for l in [1, 2, 3]:
                s = popdf[(popdf.Method == m) & (popdf.Type == t) & (popdf.Level == l)]
                c, n = s.iou_c_rel_90.values, s.iou_n_rel_90.values
                ce = ST.boot_ci_stat(c, B=B, seed=SEED); ne = ST.boot_ci_stat(n, B=B, seed=SEED)
                ratio = ST.ratio_of_means_ci(n, c, B=B, seed=SEED)
                rows.append(dict(population=popname, method=m, type=t, level=l, n=len(s),
                                 clean_iou=ce[0], clean_ci_low=ce[1], clean_ci_high=ce[2],
                                 noisy_iou=ne[0], noisy_ci_low=ne[1], noisy_ci_high=ne[2],
                                 rel_drop=1 - ratio[0], rel_drop_ci_low=1 - ratio[2], rel_drop_ci_high=1 - ratio[1]))
T7 = pd.DataFrame(rows); save_table(T7, "T7_iou_under_perturbation")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(T7[T7.population == "TP"][["method", "type", "level", "clean_iou", "noisy_iou", "rel_drop", "rel_drop_ci_low", "rel_drop_ci_high"]].to_string(index=False))

### Figure F5 · Localization (Rel-90 IoU) of correctly classified positives under perturbation

Level 0 is the clean image. Mean IoU with 95% bootstrap CI band; one panel per perturbation type.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.7), sharey=True)
tp7 = T7[T7.population == "TP"]
for a, t, letter in zip(ax, S.TYPES, "abc"):
    for m in S.METHODS:
        s = tp7[(tp7.method == m) & (tp7.type == t)].sort_values("level")
        c0 = s.iloc[0]
        x = np.r_[0, s.level]; yv = np.r_[c0.clean_iou, s.noisy_iou]
        lo = np.r_[c0.clean_ci_low, s.noisy_ci_low]; hi = np.r_[c0.clean_ci_high, s.noisy_ci_high]
        a.fill_between(x, lo, hi, color=S.METHOD_COLORS[m], alpha=0.18, lw=0)
        a.plot(x, yv, marker=S.METHOD_MARKERS[m], color=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m])
    a.set_xticks([0, 1, 2, 3], ["clean", "L1", "L2", "L3"]); a.set_xlabel(S.TYPE_LABELS[t]); a.grid(axis="x", visible=False)
    S.panel_label(a, letter)
ax[0].set_ylabel("IoU with lesion box (Rel-90)"); ax[0].set_ylim(0, None); ax[0].legend(loc="lower left")
fig.subplots_adjust(wspace=0.12, bottom=0.22)
save_fig(fig, "F5_iou_under_perturbation", tp7)
print("saved F5_iou_under_perturbation")

## 6 · The percentile-based false activation rate is definitional (R2-M2)

In v1 the FAR of a **negative** image is the number of pixels above the heatmap's own 90th percentile divided by all pixels, with no mask; this equals 0.1 for any heatmap with few ties. For positives it is the share of non-box pixels among those top-10% pixels, normalised by the non-box area, which is bounded by about 0.1 / (1 - box fraction).
**T8** shows the distribution; **F6** shows it. In v2 FAR is replaced by absolute-threshold measures (D-03).

In [ ]:
rows = []
for lab, sub in [("negative", xai[xai.Label == 0]), ("positive", xai[xai.Label == 1])]:
    for m in S.METHODS:
        v = sub[sub.Method == m].false_activation.values
        rows.append(dict(label=lab, method=m, n=len(v), mean=v.mean(), std=v.std(), min=v.min(), p01=np.quantile(v, 0.01),
                         median=np.median(v), p99=np.quantile(v, 0.99), max=v.max(), share_within_1e3_of_0p1=float((abs(v - 0.1) < 1e-3).mean())))
T8 = pd.DataFrame(rows); save_table(T8, "T8_far_percentile_audit")
with pd.option_context("display.float_format", "{:.5f}".format):
    print(T8.to_string(index=False))

### Figure F6 · Distribution of the v1 percentile FAR

(a) Negative patients: the value is fixed at 0.1 by construction; (b) positive patients. Histograms of all patient-condition rows per method.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.8))
hdata = []
for a, lab, letter in zip(ax, [0, 1], "ab"):
    sub = xai[xai.Label == lab]
    bins = np.linspace(0.0, 0.18, 73)
    for m in S.METHODS:
        h, _ = np.histogram(sub[sub.Method == m].false_activation, bins=bins)
        a.stairs(h / h.sum(), bins, color=S.METHOD_COLORS[m], lw=1.6, label=S.METHOD_LABELS[m])
        hdata.append(pd.DataFrame({"label": "negative" if lab == 0 else "positive", "method": m, "bin_low": bins[:-1], "bin_high": bins[1:], "share": h / h.sum()}))
    a.axvline(0.1, color=S.INK, lw=1, ls=":")
    a.set(xlabel=f"Percentile-based FAR, {'negative' if lab == 0 else 'positive'} patients", ylabel="Share of rows")
    S.panel_label(a, letter)
ax[0].set_yscale("log"); ax[0].legend(loc="upper left")
fig.subplots_adjust(wspace=0.3, bottom=0.22)
save_fig(fig, "F6_far_percentile_audit", pd.concat(hdata, ignore_index=True))
print("saved F6_far_percentile_audit")

## 7 · Component redundancy and validity against localization change (A-04, A-05)

- **T9**: Pearson and Spearman correlations between XFI, its three components (heatmap SSIM, Pearson r, Spearman r), the clean IoU used in the v1 Figure 8, and the change in localization caused by the perturbation (delta IoU = clean - noisy, and its absolute value), on positive patient-condition rows, per method.
- **T9b**: mean absolute delta IoU per XFI decile, per method, with bootstrap CI.

The v1 manuscript used the correlation with heatmap SSIM (an XFI component) as validation and the correlation with clean IoU as evidence of independence; neither is a valid test. Input-image SSIM/PSNR is not in the v1 data; it is added in v2 (D-17).

In [ ]:
pos["d_iou"] = pos.iou_c_rel_90 - pos.iou_n_rel_90
pos["abs_d_iou"] = pos.d_iou.abs()
vars_ = {"xfi": "XFI", "ssim": "Heatmap SSIM", "p_corr": "Pearson r", "s_corr": "Spearman r",
         "iou_c_rel_90": "Clean IoU", "d_iou": "ΔIoU", "abs_d_iou": "|ΔIoU|"}
rows = []
for m in S.METHODS + ["both"]:
    s = pos if m == "both" else pos[pos.Method == m]
    for kind in ["pearson", "spearman"]:
        c = s[list(vars_)].corr(method=kind)
        for i in vars_:
            for j in vars_:
                rows.append(dict(method=m, kind=kind, var1=i, var2=j, r=c.loc[i, j]))
T9 = pd.DataFrame(rows); save_table(T9, "T9_correlations")

rows = []
for m in S.METHODS:
    s = pos[pos.Method == m].copy(); s["dec"] = pd.qcut(s.xfi, 10, labels=False, duplicates="drop")
    for d, g in s.groupby("dec"):
        e, lo, hi = ST.boot_ci_stat(g.abs_d_iou.values, B=B, seed=SEED)
        rows.append(dict(method=m, decile=int(d) + 1, n=len(g), xfi_mean=g.xfi.mean(), abs_d_iou=e, ci_low=lo, ci_high=hi))
T9b = pd.DataFrame(rows); save_table(T9b, "T9b_abs_delta_iou_by_xfi_decile")
key = T9[(T9.kind == "spearman") & (T9.var1 == "xfi") & (T9.var2.isin(["ssim", "iou_c_rel_90", "d_iou", "abs_d_iou"]))]
print(key.pivot_table(index="method", columns="var2", values="r").round(3).to_string())

### Figure F7 · What XFI measures

(a) Spearman correlation matrix (both methods pooled, positive patients). (b) Mean absolute change in Rel-90 IoU against XFI decile, per method, with 95% bootstrap CIs: more fragile heatmaps go together with larger localization changes.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.3), gridspec_kw={"width_ratios": [1.05, 1]})
a = ax[0]
cm = T9[(T9.method == "both") & (T9.kind == "spearman")].pivot(index="var1", columns="var2", values="r").loc[list(vars_), list(vars_)]
im = a.imshow(cm.values, cmap=S.DIVERGING, vmin=-1, vmax=1)
for i in range(len(vars_)):
    for j in range(len(vars_)):
        v = cm.values[i, j]
        a.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if abs(v) > 0.6 else S.INK)
a.set_xticks(range(len(vars_)), list(vars_.values()), rotation=45, ha="right"); a.set_yticks(range(len(vars_)), list(vars_.values()))
a.grid(False); [s.set_visible(False) for s in a.spines.values()]
cb = fig.colorbar(im, ax=a, fraction=0.046, pad=0.03); cb.set_label("Spearman ρ", labelpad=2); cb.outline.set_visible(False)
S.panel_label(a, "a")
a = ax[1]
for m in S.METHODS:
    s = T9b[T9b.method == m]
    a.errorbar(s.decile, s.abs_d_iou, yerr=[s.abs_d_iou - s.ci_low, s.ci_high - s.abs_d_iou], marker=S.METHOD_MARKERS[m],
               color=S.METHOD_COLORS[m], capsize=2, lw=1.5, ms=5, label=S.METHOD_LABELS[m])
a.set(xlabel="XFI decile (1 = least fragile)", ylabel="Mean |ΔIoU| (Rel-90)", ylim=(0, None)); a.set_xticks(range(1, 11))
a.legend(loc="upper left"); S.panel_label(a, "b")
fig.subplots_adjust(wspace=0.75, bottom=0.25)
save_fig(fig, "F7_validity", {"spearman_matrix": cm.reset_index().rename(columns={"var1": "variable"}), "deciles": T9b})
print("saved F7_validity")

## 8 · Baselines (A-06)

**T10** describes the two v1 baselines. The uniform map has zero variance, so SSIM is set to 0 and the undefined correlations are replaced by 0.5 after normalisation; its XFI of 2/3 is therefore a convention, not a measurement. The random baseline compares two independent pixel-noise maps; 2/3 is the expected XFI of uncorrelated maps. New baselines are added in v2 (D-14).

In [ ]:
rows = []
for m in ["Random_Baseline", "Uniform_Baseline"]:
    s = df[df.Method == m]
    rows.append(dict(method=m, n=len(s), xfi_mean=s.xfi.mean(), xfi_std=s.xfi.std(), xfi_min=s.xfi.min(), xfi_max=s.xfi.max(),
                     ssim_mean=s.ssim.mean(), p_corr_nan_share=float(s.p_corr.isna().mean()), s_corr_nan_share=float(s.s_corr.isna().mean())))
T10 = pd.DataFrame(rows); save_table(T10, "T10_baselines")
print(T10.to_string(index=False))

## 9 · Manuscript numbers against the v1 data (A-01)

**T11** lists each number reported in the submitted manuscript next to the value recomputed from ART-001 with the manuscript's own (row-level) method, and, where relevant, the patient-level value from this notebook. It is the basis of the transparency paragraph in the response letter (D-23).

In [ ]:
def rowlevel_slope(m):
    s = xai[xai.Method == m].groupby("Level").xfi.mean(); return float(np.polyfit(s.index, s.values, 1)[0])
def r_rowlevel(metric):
    q = xai.pivot_table(index=["Patient", "Type", "Level"], columns="Method", values=metric)
    return ST.wilcoxon_paired(q.GradCAM.values, q.EigenCAM.values)["r"]
cl = clean.groupby("Method").iou_c_rel_90.mean()
far = xai.groupby("Method").false_activation.mean()
mx = xai.groupby("Method").xfi.mean()
ov = T4.iloc[0]
T11 = pd.DataFrame([
    dict(item="Mean XFI, Grad-CAM", manuscript=0.2072, v1_data=mx.GradCAM, patient_level=pp.GradCAM.mean()),
    dict(item="Mean XFI, Eigen-CAM", manuscript=0.2694, v1_data=mx.EigenCAM, patient_level=pp.EigenCAM.mean()),
    dict(item="Degradation slope, Grad-CAM", manuscript=-0.0741, v1_data=rowlevel_slope("GradCAM"), patient_level=T6.set_index("type").loc["pooled", "GradCAM_slope"]),
    dict(item="Degradation slope, Eigen-CAM", manuscript=-0.0580, v1_data=rowlevel_slope("EigenCAM"), patient_level=T6.set_index("type").loc["pooled", "EigenCAM_slope"]),
    dict(item="Effect size r (XFI)", manuscript=0.2552, v1_data=r_rowlevel("xfi"), patient_level=ov.r),
    dict(item="Effect size r (SSIM)", manuscript=0.3535, v1_data=r_rowlevel("ssim"), patient_level=np.nan),
    dict(item="Clean Rel-90 IoU, Grad-CAM", manuscript=0.1066, v1_data=cl.GradCAM, patient_level=cl.GradCAM),
    dict(item="Clean Rel-90 IoU, Eigen-CAM", manuscript=0.0566, v1_data=cl.EigenCAM, patient_level=cl.EigenCAM),
    dict(item="FAR, Grad-CAM", manuscript=0.0977, v1_data=far.GradCAM, patient_level=np.nan),
    dict(item="FAR, Eigen-CAM", manuscript=0.0998, v1_data=far.EigenCAM, patient_level=np.nan),
    dict(item="Random baseline XFI", manuscript=0.6649, v1_data=df[df.Method == "Random_Baseline"].xfi.mean(), patient_level=np.nan),
    dict(item="Uniform baseline XFI", manuscript=0.6667, v1_data=df[df.Method == "Uniform_Baseline"].xfi.mean(), patient_level=np.nan),
])
T11["matches"] = (T11.manuscript - T11.v1_data).abs() < 5e-4
save_table(T11, "T11_manuscript_vs_v1")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(T11.to_string(index=False))

## Summary, artifacts and LOG entry

Writes `output/phase1/artifacts_phase1.csv` (every file produced here with its sha256) and prints the lines to append to `_state/ARTIFACTS.csv` and `_state/LOG.md`. Paste this cell's output back into the chat.

In [ ]:
from datetime import date
arts = pd.DataFrame([dict(path=str(p.relative_to(ROOT)), bytes=p.stat().st_size, sha256=sha256(p)) for p in PRODUCED])
arts.to_csv(OUT / "artifacts_phase1.csv", index=False)
print(f"{len(arts)} files written ({arts.bytes.sum()/1e6:.1f} MB)")

ov = T4.iloc[0]; c5 = T1.set_index(["metric", "threshold"])
print("\n--- ARTIFACTS line (append to _state/ARTIFACTS.csv) ---")
print(f"ART-010,{date.today()},output/phase1/artifacts_phase1.csv,table,1,notebooks/01_csv_audit.ipynb,{sha256(OUT / 'artifacts_phase1.csv')},"
      f"Phase 1 audit outputs on ART-001 ({len(arts)} files: tables T1-T11 and figures F1-F7; hashes inside)")
print("\n--- LOG entry (append to _state/LOG.md) ---")
print(f"## {date.today()} | Phase 1: audit of v1 results (01_csv_audit)")
print(f"- Model A test: AUC {c5.loc[('AUC','threshold-free'),'estimate']:.3f} "
      f"[{c5.loc[('AUC','threshold-free'),'ci_low']:.3f}, {c5.loc[('AUC','threshold-free'),'ci_high']:.3f}]; "
      f"recall@0.5 {c5.loc[('recall','0.5'),'estimate']:.3f}; F1@0.5 {c5.loc[('f1','0.5'),'estimate']:.3f}; "
      f"ECE {c5.loc[('ECE_10bins','threshold-free'),'estimate']:.3f}; Brier {c5.loc[('Brier','threshold-free'),'estimate']:.3f}; "
      f"Youden (post hoc) {thr_y:.3f}: recall {c5.loc[('recall','youden_posthoc'),'estimate']:.3f}.")
t2 = T2[T2.iou == "Rel-90"].set_index(["method", "group"])["mean"]
print(f"- Clean Rel-90 IoU TP/FN: Grad {t2[('GradCAM','TP')]:.3f}/{t2[('GradCAM','FN')]:.3f}, Eigen {t2[('EigenCAM','TP')]:.3f}/{t2[('EigenCAM','FN')]:.3f}.")
print(f"- Patient-level Grad minus Eigen mean XFI: median {ov.median_diff:+.4f} [{ov.median_ci_low:+.4f}, {ov.median_ci_high:+.4f}], p {ov.p:.3g}, r {ov.r:.3f}.")
print(f"- Manuscript numbers matching v1 data: {int(T11.matches.sum())}/{len(T11)} (only baselines).")
print(f"- Outputs: {len(arts)} files in output/phase1 and fig/phase1_v1 (ART-010).")